# Phase 3 — EV Bus Video Analytics (Colab)

**Project:** Multimodal AI Strategy for EV-Bus Market Disruption in India  
**Notebook:** `05_video_analytics_colab.ipynb`  
**Self-contained:** No external library files required — run all cells top to bottom.

### Videos analysed
| Brand | Type |
|-------|------|
| EKA | New-age OEM |
| OLECTRA | New-age OEM |
| SWITCH | New-age OEM |
| TATA | Legacy OEM |

### Pipeline stages
1. Frame extraction (OpenCV, 1 fps)
2. Low-level visual features (brightness, contrast, saturation, sharpness, edge density, colourfulness, visual balance)
3. Shot-boundary detection (inter-frame pixel difference)
4. OCR text extraction (Tesseract)
5. VADER sentiment on OCR text
6. EV-bus keyword theme tagging
7. DETR object detection · BLIP captions · CLIP zero-shot *(torch / GPU)*
8. Per-video aggregation and brand comparison
9. Charts saved to Drive

### Methodology notes
- All findings are **exploratory** — 4 videos is not a statistically valid sample.
- OCR sentiment reflects on-screen text framing, not passenger/operator experience.
- Video production quality ≠ bus manufacturing quality.
- Correlation ≠ causality. Label small-sample findings as exploratory in the report.

## Step 0 — Install dependencies
Run once per Colab session (runtime restart not required).

In [ ]:
# System-level: Tesseract OCR engine
!apt-get install -y -q tesseract-ocr 2>/dev/null

# Python packages
!pip install -q \
    pytesseract \
    vaderSentiment \
    wordcloud \
    scikit-learn \
    opencv-python-headless

# torch + transformers are pre-installed in Colab — just confirm:
import importlib
for pkg in ["torch", "transformers", "cv2", "pytesseract", "vaderSentiment"]:
    status = "✓" if importlib.util.find_spec(pkg) else "✗ MISSING"
    print(f"  {status}  {pkg}")
print("\nAll done — proceed to Step 1.")

## Step 1 — Mount Google Drive and set paths

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

# -----------------------------------------------------------------------
# EDIT THIS BLOCK if your folder is in a different Drive location
# -----------------------------------------------------------------------
DRIVE_ROOT      = Path("/content/drive/MyDrive/Phase_3_Video_Analytics")
VIDEOS_DIR      = DRIVE_ROOT / "Videos"
OUTPUT_ROOT     = DRIVE_ROOT / "Video_Analytics" / "outputs"
# -----------------------------------------------------------------------

FRAMES_DIR    = OUTPUT_ROOT / "frames"
PERFRAME_DIR  = OUTPUT_ROOT / "per_frame"
SUMMARIES_DIR = OUTPUT_ROOT / "summaries"
CHARTS_DIR    = OUTPUT_ROOT / "charts"

for d in [FRAMES_DIR, PERFRAME_DIR, SUMMARIES_DIR, CHARTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

video_files = sorted(VIDEOS_DIR.glob("*.mp4"))
print(f"Videos directory : {VIDEOS_DIR}")
print(f"Outputs directory: {OUTPUT_ROOT}")
print(f"Videos found     : {[v.name for v in video_files]}")

if not video_files:
    raise FileNotFoundError(
        f"No .mp4 files found in {VIDEOS_DIR}.\n"
        "Check that Phase_3_Video_Analytics/Videos/ exists in your Drive."
    )

## Step 2 — Pipeline configuration
Adjust `SAMPLE_FPS` and `MAX_FRAMES` based on your available time/compute.

| Setting | CPU (free) | GPU (T4) |
|---------|-----------|----------|
| `SAMPLE_FPS=0.5, MAX_FRAMES=60` | ~8 min | ~3 min |
| `SAMPLE_FPS=1.0, MAX_FRAMES=120` | ~20 min | ~7 min |
| `SAMPLE_FPS=1.0, MAX_FRAMES=200` | ~35 min | ~12 min |

In [ ]:
SAMPLE_FPS = 1.0    # frames sampled per second of video
MAX_FRAMES = 120    # hard cap per video
USE_DEEP_MODELS = True  # set False to skip BLIP/DETR/CLIP (much faster)

## Step 3 — Library code (all inline)
Self-contained — no external `.py` files needed.

In [ ]:
import json
import math
import os
import re
import time
import warnings
from collections import Counter, defaultdict
from datetime import datetime
from functools import lru_cache
from typing import Optional

import cv2
import numpy as np
import pandas as pd
from PIL import Image

%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
matplotlib.rcParams["figure.dpi"] = 110
matplotlib.rcParams["font.size"] = 10

warnings.filterwarnings("ignore")

# Check torch
try:
    import torch
    _TORCH_AVAILABLE = USE_DEEP_MODELS
    if _TORCH_AVAILABLE:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"torch {torch.__version__}  |  device: {device}")
    else:
        device = "cpu"
        print("torch found but USE_DEEP_MODELS=False — skipping BLIP/DETR/CLIP")
except ImportError:
    _TORCH_AVAILABLE = False
    device = "cpu"
    print("torch not found — running OpenCV + OCR only")

print(f"Deep models: {'ENABLED' if _TORCH_AVAILABLE else 'DISABLED'}")

In [ ]:
# ── Brand colours ────────────────────────────────────────────────────────────
BRAND_COLORS = {
    "EKA":     "#1a73e8",
    "OLECTRA": "#34a853",
    "SWITCH":  "#fbbc04",
    "TATA":    "#ea4335",
}
def _brand_color(brand): return BRAND_COLORS.get(brand.upper(), "#888888")

# ── EV-bus keyword theme dictionary ──────────────────────────────────────────
_EV_BUS_THEME_KEYWORDS = {
    "battery_technology":  {"battery","kwh","range","charge","charging","lithium",
                            "energy","power","storage","cell"},
    "safety":              {"safety","safe","adas","brake","camera","alert",
                            "emergency","airbag","sensor"},
    "passenger_comfort":   {"comfort","seat","ac","interior","space","passenger",
                            "legroom","wifi","usb","smooth"},
    "technology":          {"technology","tech","ai","smart","digital","connected",
                            "software","system","iot","automation"},
    "sustainability":      {"green","electric","emission","clean","eco","zero",
                            "sustainable","environment","carbon","renewable"},
    "reliability":         {"reliable","reliability","uptime","service","maintain",
                            "warranty","durable","performance","proven","tested"},
    "cost_economics":      {"cost","price","saving","efficient","economy","per",
                            "km","kilometre","affordable","total"},
    "govt_policy":         {"government","ministry","scheme","tender","subsidy",
                            "policy","national","india","state","pmgsy"},
    "brand_trust":         {"trust","quality","certified","iso","partner","global",
                            "experience","years","leader","best"},
    "fleet_operations":    {"fleet","depot","route","operator","schedule","city",
                            "transport","bus","buses","network"},
}

_STOP_WORDS = {
    "the","and","for","with","from","you","are","this","that","they","their",
    "has","have","been","not","but","all","any","was","were","its","can","will",
    "more","than","into","who","how","what","why","when","our","your","new","now",
    "get","www","http","https","com","also","just","one","two",
}

# ── CLIP prompt banks ────────────────────────────────────────────────────────
CLIP_BINARY_VIDEO = {
    "hl_bus_exterior":      ("an electric bus exterior driving on a city road",
                             "an image with no bus exterior visible"),
    "hl_bus_interior":      ("the interior of an electric bus with seats and passengers",
                             "an exterior shot or non-bus image"),
    "hl_charging_infra":    ("electric bus charging station or charging infrastructure",
                             "a bus with no charging equipment visible"),
    "hl_fleet_scale":       ("a large fleet of many electric buses lined up together",
                             "a single bus or no bus visible"),
    "hl_govt_ceremony":     ("a government or corporate handover ceremony for new buses",
                             "an ordinary bus video with no ceremony"),
    "hl_tech_showcase":     ("advanced technology dashboard screens battery systems in a bus",
                             "a plain bus interior with no technology emphasis"),
    "hl_passenger_comfort": ("comfortable passengers enjoying a smooth electric bus ride",
                             "an empty bus or non-passenger content"),
    "hl_operational_proof": ("real-world footage of buses in daily passenger service",
                             "a studio marketing render or product demo with no real service"),
    "hl_brand_marketing":   ("a polished brand marketing or advertisement video for a bus",
                             "raw documentary footage with no marketing production"),
}
CLIP_SETTING_VIDEO = [
    "urban city road or highway",
    "rural highway or expressway",
    "bus depot or maintenance yard",
    "studio or product launch event",
    "government or corporate ceremony",
]
CLIP_THEMES_VIDEO = {
    "theme_technology":     ("emphasising advanced technology innovation and electronics",
                             "no emphasis on technology"),
    "theme_comfort":        ("showcasing passenger comfort spacious seats and smooth ride",
                             "not about comfort"),
    "theme_sustainability": ("green eco-friendly zero-emission clean energy message",
                             "no sustainability or green theme"),
    "theme_reliability":    ("demonstrating reliability uptime and dependable service",
                             "no reliability or uptime message"),
    "theme_safety":         ("emphasising safety security and passenger protection",
                             "no safety message"),
}
CLIP_PERCEPTION_VIDEO = {
    "pxy_trust":             ("a trustworthy well-established reliable bus brand",
                              "an untrustworthy or unreliable bus brand"),
    "pxy_modernity":         ("a cutting-edge modern futuristic electric bus",
                              "an outdated conventional diesel bus"),
    "pxy_comfort":           ("a comfortable spacious premium bus experience",
                              "an uncomfortable cramped bus"),
    "pxy_sustainability":    ("a clean zero-emission environmentally friendly electric bus",
                              "a polluting diesel or petrol bus"),
    "pxy_operational_scale": ("a proven bus deployed at scale serving many passengers",
                              "a prototype or limited-deployment bus"),
    "pxy_visual_production": ("a high-production-quality professional brand video",
                              "a low-quality amateur or raw video"),
}
print("Constants loaded.")

In [ ]:
# ── Lazy model loaders (only download if _TORCH_AVAILABLE) ───────────────────
_BLIP_MODEL = None
_DETR_MODEL = None
_CLIP_MODEL = None
_CLIP_TEXT_BANK = None

def _load_blip():
    global _BLIP_MODEL
    if _BLIP_MODEL is None and _TORCH_AVAILABLE:
        from transformers import BlipProcessor, BlipForConditionalGeneration
        print("  Loading BLIP…", end=" ", flush=True)
        proc = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
        mdl  = BlipForConditionalGeneration.from_pretrained(
                   "Salesforce/blip-image-captioning-base").to(device).eval()
        _BLIP_MODEL = (proc, mdl)
        print("done")
    return _BLIP_MODEL

def _load_detr():
    global _DETR_MODEL
    if _DETR_MODEL is None and _TORCH_AVAILABLE:
        from transformers import DetrImageProcessor, DetrForObjectDetection
        print("  Loading DETR…", end=" ", flush=True)
        proc = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
        mdl  = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50").to(device).eval()
        _DETR_MODEL = (proc, mdl)
        print("done")
    return _DETR_MODEL

def _load_clip():
    global _CLIP_MODEL, _CLIP_TEXT_BANK
    if _CLIP_MODEL is None and _TORCH_AVAILABLE:
        from transformers import CLIPProcessor, CLIPModel
        print("  Loading CLIP…", end=" ", flush=True)
        proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
        mdl  = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
        _CLIP_MODEL = (proc, mdl)
        # Pre-encode all text prompts once
        prompts, slices = [], {}
        i = 0
        for name, (pos, neg) in {**CLIP_BINARY_VIDEO, **CLIP_THEMES_VIDEO,
                                   **CLIP_PERCEPTION_VIDEO}.items():
            slices[name] = ("pair", i); prompts += [pos, neg]; i += 2
        slices["_setting"] = ("multi", i); prompts += CLIP_SETTING_VIDEO
        with torch.no_grad():
            tok = proc(text=prompts, return_tensors="pt", padding=True).to(device)
            tf  = mdl.get_text_features(**tok)
            if hasattr(tf, "pooler_output"): tf = tf.pooler_output
            tf  = tf / tf.norm(dim=-1, keepdim=True)
        scale = float(mdl.logit_scale.exp())
        _CLIP_TEXT_BANK = (tf, slices, scale)
        print("done")
    return _CLIP_MODEL, _CLIP_TEXT_BANK

print("Model loaders defined.")

In [ ]:
# ── Low-level visual feature functions ───────────────────────────────────────
def _warm_cool_bias(rgb):
    r,g,b = rgb[...,0].mean(), rgb[...,1].mean(), rgb[...,2].mean()
    warm = (r+0.5*g)/255.0; cool = (b+0.5*g)/255.0
    return round((warm-cool)/(warm+cool),3) if (warm+cool)>0 else 0.0

def _colourfulness(rgb):
    R=rgb[...,0].astype(float); G=rgb[...,1].astype(float); B=rgb[...,2].astype(float)
    rg=R-G; yb=0.5*(R+G)-B
    return round(math.sqrt(rg.std()**2+yb.std()**2)+0.3*math.sqrt(rg.mean()**2+yb.mean()**2),2)

def _edge_density(gray):
    return round(float((cv2.Canny(gray,100,200)>0).mean()),4)

def _gradient_energy(gray):
    return np.hypot(cv2.Sobel(gray,cv2.CV_64F,1,0), cv2.Sobel(gray,cv2.CV_64F,0,1))

def _visual_balance(gray):
    e=_gradient_energy(gray)+1e-9; h,w=gray.shape
    L=e[:,:w//2].sum(); R=e[:,w-w//2:].sum()
    T=e[:h//2,:].sum(); B=e[h-h//2:,:].sum()
    return round(float((1-abs(L-R)/(L+R)+1-abs(T-B)/(T+B))/2),3)

def _figure_ground(gray):
    e=_gradient_energy(gray); h,w=gray.shape
    cy0,cy1=int(h*.25),int(h*.75); cx0,cx1=int(w*.25),int(w*.75)
    ctr=e[cy0:cy1,cx0:cx1].mean()
    msk=np.ones((h,w),bool); msk[cy0:cy1,cx0:cx1]=False
    brd=e[msk].mean()
    return round(float(ctr/(ctr+brd+1e-9)),3)

def low_level_features(frame_path):
    bgr = cv2.imread(str(frame_path))
    if bgr is None: return None
    h,w = bgr.shape[:2]
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    hsv  = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    rgb  = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    return {
        "width":w, "height":h,
        "brightness":   round(float(gray.mean()),2),
        "contrast":     round(float(gray.std()),2),
        "saturation":   round(float(hsv[...,1].mean()),2),
        "sharpness":    round(float(cv2.Laplacian(gray,cv2.CV_64F).var()),2),
        "warm_cool_bias": _warm_cool_bias(rgb),
        "colourfulness":  _colourfulness(rgb),
        "edge_density":   _edge_density(gray),
        "visual_balance": _visual_balance(gray),
        "figure_ground_sep": _figure_ground(gray),
    }

# ── Shot boundary detection ───────────────────────────────────────────────────
def detect_shot_boundaries(frame_paths, threshold_pct=90.0):
    diffs, prev_gray = [], None
    for p in frame_paths:
        bgr = cv2.imread(str(p))
        if bgr is None: diffs.append(0.0); continue
        gray = cv2.resize(cv2.cvtColor(bgr,cv2.COLOR_BGR2GRAY),(160,90))
        if prev_gray is None:
            diffs.append(0.0)
        else:
            diffs.append(float(np.abs(gray.astype(float)-prev_gray.astype(float)).mean()))
        prev_gray = gray
    arr = np.array(diffs)
    thr = np.percentile(arr[1:], threshold_pct) if len(arr)>1 else 0
    return [i for i,d in enumerate(arr) if i>0 and d>=thr], arr.tolist()

# ── OCR + VADER ───────────────────────────────────────────────────────────────
def ocr_frame(pil):
    try:
        import pytesseract
        return pytesseract.image_to_string(pil.convert("RGB")).strip()
    except Exception:
        return ""

def vader_sentiment(text):
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    if not text or not text.strip():
        return {"nf_neg":None,"nf_neu":None,"nf_pos":None,"nf_compound":None,"nf_label":"no_text"}
    s = SentimentIntensityAnalyzer().polarity_scores(text)
    lbl = "positive" if s["compound"]>=0.05 else "negative" if s["compound"]<=-0.05 else "neutral"
    return {"nf_neg":s["neg"],"nf_neu":s["neu"],"nf_pos":s["pos"],"nf_compound":s["compound"],"nf_label":lbl}

# ── Deep model inference (torch) ─────────────────────────────────────────────
_DETR_KEEP = {"bus","person","truck","car","train","traffic light","bicycle","motorcycle"}

def detect_objects(pil):
    if not _TORCH_AVAILABLE: return Counter()
    result = _load_detr()
    if result is None: return Counter()
    proc, mdl = result
    inputs = proc(images=pil, return_tensors="pt").to(device)
    with torch.no_grad(): out = mdl(**inputs)
    sizes = torch.tensor([pil.size[::-1]])
    res = proc.post_process_object_detection(out,target_sizes=sizes,threshold=0.7)[0]
    labels = [mdl.config.id2label[int(l)] for l in res["labels"]]
    return Counter(l for l in labels if l in _DETR_KEEP)

def blip_caption(pil):
    if not _TORCH_AVAILABLE: return ""
    result = _load_blip()
    if result is None: return ""
    proc, mdl = result
    inputs = proc(pil, return_tensors="pt").to(device)
    with torch.no_grad(): out = mdl.generate(**inputs, max_new_tokens=40)
    return proc.decode(out[0], skip_special_tokens=True)

def clip_features(pil):
    if not _TORCH_AVAILABLE: return {}
    (proc, mdl), (tf, slices, scale) = _load_clip()
    if proc is None: return {}
    inputs = proc(images=pil, return_tensors="pt").to(device)
    with torch.no_grad():
        vf = mdl.get_image_features(**inputs)
        if hasattr(vf,"pooler_output"): vf = vf.pooler_output
        vf = vf / vf.norm(dim=-1,keepdim=True)
        sims = (scale*(vf@tf.T)[0]).cpu()
    out = {}
    for name,(kind,i) in slices.items():
        if kind=="pair":
            p = torch.softmax(sims[i:i+2],dim=0)[0].item()
            if name.startswith("pxy_"): out[name]=round(p*100,1)
            else: out[name]=round(p,3); out[name+"_flag"]=bool(p>=0.5)
    kind,i = slices["_setting"]
    seg = torch.softmax(sims[i:i+len(CLIP_SETTING_VIDEO)],dim=0)
    labels=["urban_road","rural_highway","depot_yard","studio_launch","govt_ceremony"]
    out["hl_setting"]=labels[int(seg.argmax())]
    out["hl_setting_conf"]=round(float(seg.max()),3)
    return out

# ── Keyword theme extraction ──────────────────────────────────────────────────
def extract_text_themes(ocr_texts):
    combined = " ".join(t.lower() for t in ocr_texts if isinstance(t,str))
    tokens = set(re.sub(r"[^a-z\s]"," ",combined).split())
    return {theme: len(kws & tokens) for theme,kws in _EV_BUS_THEME_KEYWORDS.items()}

print("Feature functions loaded.")

In [ ]:
# ── Video metadata ────────────────────────────────────────────────────────────
def get_video_metadata(video_path):
    cap = cv2.VideoCapture(str(video_path))
    fps   = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    dur = total/fps if fps>0 else 0
    return {
        "video_file": Path(video_path).name,
        "brand":      Path(video_path).stem.upper(),
        "fps": round(fps,2), "total_frames": total,
        "width":w, "height":h, "resolution":f"{w}x{h}",
        "duration_seconds": round(dur,2),
        "duration_minutes": round(dur/60,2),
        "file_size_mb": round(Path(video_path).stat().st_size/(1024**2),2),
        "aspect_ratio": round(w/h,3) if h>0 else None,
    }

# ── Frame extraction ──────────────────────────────────────────────────────────
def extract_frames(video_path, output_dir, sample_fps=1.0, max_frames=200):
    output_dir = Path(output_dir); output_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    vfps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    interval = max(1, int(round(vfps/sample_fps)))
    saved, idx = [], 0
    while cap.isOpened() and len(saved)<max_frames:
        ret, frame = cap.read()
        if not ret: break
        if idx%interval==0:
            p = output_dir/f"frame_{idx:06d}.jpg"
            cv2.imwrite(str(p), frame); saved.append(p)
        idx+=1
    cap.release()
    return sorted(saved)

# ── Per-video pipeline ────────────────────────────────────────────────────────
def process_video(video_path, frames_dir, sample_fps=1.0, max_frames=200, verbose=True):
    video_path = Path(video_path)
    brand = video_path.stem.upper()
    meta  = get_video_metadata(video_path)
    if verbose:
        print(f"\n{'='*55}")
        print(f"  {brand}  |  {meta['duration_seconds']}s  |  {meta['resolution']}  |  {meta['file_size_mb']} MB")
        print(f"{'='*55}")

    fps_list = extract_frames(video_path, frames_dir, sample_fps, max_frames)
    if verbose: print(f"  [1/4] {len(fps_list)} frames extracted")

    boundaries, pixel_diffs = detect_shot_boundaries(fps_list)
    if verbose: print(f"  [2/4] {len(boundaries)} shot cuts detected")

    rows = []
    for idx, fp in enumerate(fps_list):
        ll = low_level_features(fp)
        if ll is None: continue
        pil = Image.open(fp).convert("RGB")
        ocr  = ocr_frame(pil)
        sent = vader_sentiment(ocr)
        obj  = detect_objects(pil)  if _TORCH_AVAILABLE else Counter()
        cap  = blip_caption(pil)    if _TORCH_AVAILABLE else ""
        clf  = clip_features(pil)   if _TORCH_AVAILABLE else {}
        rec  = {
            "video_file":  video_path.name, "brand": brand,
            "frame_path":  str(fp), "frame_index": idx,
            "timestamp_s": round(idx/sample_fps, 2),
            "is_shot_cut": idx in boundaries,
            "pixel_diff":  round(pixel_diffs[idx] if idx<len(pixel_diffs) else 0, 3),
            **ll,
            "ocr_text":    ocr, "ocr_char_len": len(ocr),
            **sent,
            "blip_caption": cap,
            "obj_bus":    obj.get("bus",0),   "obj_person": obj.get("person",0),
            "obj_truck":  obj.get("truck",0), "obj_car":    obj.get("car",0),
            "obj_all":    json.dumps(dict(obj)),
        }
        rec.update(clf)
        rows.append(rec)
        if verbose and (idx%25==0 or idx==len(fps_list)-1):
            txt = ocr[:35].replace("\n"," ") if ocr else "-"
            print(f"    frame {idx:>4}/{len(fps_list)-1}  "
                  f"bright={ll['brightness']:5.1f}  ocr='{txt}'")

    df = pd.DataFrame(rows)
    for k,v in meta.items(): df[f"meta_{k}"] = v
    if verbose: print(f"  [4/4] {len(df)} frame records built")
    return df

# ── Per-video summary ─────────────────────────────────────────────────────────
def summarise_video(df):
    if df.empty: return {}
    brand = df["brand"].iloc[0]
    meta  = {c.replace("meta_",""): df[c].iloc[0] for c in df.columns if c.startswith("meta_")}
    vis   = ["brightness","contrast","saturation","sharpness",
             "warm_cool_bias","colourfulness","edge_density","visual_balance","figure_ground_sep"]
    avgs  = {f"avg_{c}": round(float(df[c].mean()),3) for c in vis if c in df.columns}
    cuts  = int(df["is_shot_cut"].sum())
    dur   = meta.get("duration_seconds",1)
    themes = extract_text_themes(df["ocr_text"].dropna().tolist())
    dominant = max(themes,key=themes.get) if themes else "none"
    ocr_f = int((df["ocr_char_len"]>10).sum())
    vdf   = df[df["nf_compound"].notna()]
    pxy   = {c: round(float(df[c].mean()),1) for c in df.columns if c.startswith("pxy_") and df[c].notna().any()}
    hlf   = {c: round(float(df[c].mean()*100),1) for c in df.columns if c.endswith("_flag") and df[c].notna().any()}
    return {
        "brand": brand, **meta,
        "total_frames_analysed": len(df),
        "shot_cuts": cuts,
        "cuts_per_minute": round(cuts/(dur/60),2) if dur>0 else 0,
        "avg_pixel_diff": round(float(df["pixel_diff"].mean()),3),
        **avgs,
        "ocr_text_frames": ocr_f,
        "ocr_text_pct": round(ocr_f/len(df)*100,1) if len(df)>0 else 0,
        "text_themes": themes,
        "dominant_text_theme": dominant,
        "avg_vader_compound": round(float(vdf["nf_compound"].mean()),3) if not vdf.empty else None,
        "sentiment_distribution": (vdf["nf_label"].value_counts(normalize=True).round(3).to_dict()
                                   if not vdf.empty else {}),
        "total_obj_bus":    int(df["obj_bus"].sum()),
        "total_obj_person": int(df["obj_person"].sum()),
        "pct_frames_obj_bus":    round(float((df["obj_bus"]>0).mean()*100),1),
        "pct_frames_obj_person": round(float((df["obj_person"]>0).mean()*100),1),
        "perception_scores": pxy,
        "hl_flag_frequencies_pct": hlf,
        "deep_models_used": _TORCH_AVAILABLE,
    }

print("Pipeline functions loaded.")

In [ ]:
# ── Chart functions ───────────────────────────────────────────────────────────

def fig_brightness_timeline(df, output_path):
    fig, ax = plt.subplots(figsize=(13,4))
    c = _brand_color(df["brand"].iloc[0])
    ax.fill_between(df["timestamp_s"], df["brightness"], alpha=0.2, color=c)
    ax.plot(df["timestamp_s"], df["brightness"], color=c, linewidth=1.2)
    cuts = df[df["is_shot_cut"]]
    ax.vlines(cuts["timestamp_s"],0,255,colors="crimson",lw=0.8,ls="--",alpha=0.6,label="Shot cut")
    ax.set(xlabel="Time (s)",ylabel="Brightness (0–255)",ylim=(0,255),
           title=f"{df['brand'].iloc[0]} — Brightness Timeline + Shot Cuts")
    ax.legend(fontsize=9); fig.tight_layout()
    fig.savefig(str(output_path),dpi=130,bbox_inches="tight"); plt.show(); plt.close(fig)

def fig_visual_radar(summaries, output_path):
    metrics = ["avg_brightness","avg_saturation","avg_colourfulness",
               "avg_edge_density","avg_visual_balance"]
    labels  = [m.replace("avg_","").replace("_"," ").title() for m in metrics]
    N = len(labels); angles = [n/N*2*math.pi for n in range(N)]+[0]
    fig, ax = plt.subplots(figsize=(7,7),subplot_kw={"polar":True})
    for s in summaries:
        brand = s["brand"]
        norms = [s.get("avg_brightness",0)/255, s.get("avg_saturation",0)/255,
                 min(s.get("avg_colourfulness",0)/80,1), min(s.get("avg_edge_density",0)/0.15,1),
                 s.get("avg_visual_balance",0)]
        vals = norms+[norms[0]]
        ax.plot(angles,vals,label=brand,color=_brand_color(brand),linewidth=2)
        ax.fill(angles,vals,alpha=0.1,color=_brand_color(brand))
    ax.set_xticks(angles[:-1]); ax.set_xticklabels(labels,size=10)
    ax.set_title("Visual Quality Radar — Brand Comparison",size=12,pad=20)
    ax.legend(loc="upper right",bbox_to_anchor=(1.3,1.1)); fig.tight_layout()
    fig.savefig(str(output_path),dpi=130,bbox_inches="tight"); plt.show(); plt.close(fig)

def fig_text_themes_heatmap(summaries, output_path):
    brands = [s["brand"] for s in summaries]
    themes = list(_EV_BUS_THEME_KEYWORDS.keys())
    mat = np.array([[s.get("text_themes",{}).get(t,0) for t in themes] for s in summaries],dtype=float)
    fig, ax = plt.subplots(figsize=(max(10,len(themes)*1.2), max(3,len(brands)*0.9)))
    im = ax.imshow(mat,aspect="auto",cmap="YlOrRd")
    ax.set_xticks(range(len(themes)))
    ax.set_xticklabels([t.replace("_","\n").title() for t in themes],rotation=0,ha="center",fontsize=9)
    ax.set_yticks(range(len(brands))); ax.set_yticklabels(brands,fontsize=11)
    for i in range(len(brands)):
        for j in range(len(themes)):
            ax.text(j,i,int(mat[i,j]),ha="center",va="center",fontsize=9,
                    color="white" if mat[i,j]>mat.max()*0.6 else "black")
    plt.colorbar(im,ax=ax,label="Keyword hits")
    ax.set_title("On-Screen Text Themes by Brand (OCR keyword count)")
    fig.tight_layout(); fig.savefig(str(output_path),dpi=130,bbox_inches="tight"); plt.show(); plt.close(fig)

def fig_shot_pacing(summaries, output_path):
    brands=[s["brand"] for s in summaries]; cpm=[s.get("cuts_per_minute",0) for s in summaries]
    fig, ax = plt.subplots(figsize=(8,5))
    bars = ax.bar(brands,cpm,color=[_brand_color(b) for b in brands],edgecolor="white")
    for bar,v in zip(bars,cpm):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                f"{v:.1f}",ha="center",va="bottom",fontsize=12,fontweight="bold")
    ax.set(ylabel="Shot cuts per minute",ylim=(0,max(cpm)*1.35+1),
           title="Editing Pacing — Shot Cuts per Minute\n(Higher = faster, dynamic edit)")
    fig.tight_layout(); fig.savefig(str(output_path),dpi=130,bbox_inches="tight"); plt.show(); plt.close(fig)

def fig_sentiment_bars(summaries, output_path):
    brands=[s["brand"] for s in summaries]
    pos=[s.get("sentiment_distribution",{}).get("positive",0) for s in summaries]
    neu=[s.get("sentiment_distribution",{}).get("neutral",0) for s in summaries]
    neg=[s.get("sentiment_distribution",{}).get("negative",0) for s in summaries]
    x=np.arange(len(brands))
    fig, ax = plt.subplots(figsize=(9,5))
    ax.bar(x,pos,0.5,label="Positive",color="#34a853")
    ax.bar(x,neu,0.5,bottom=pos,label="Neutral",color="#fbbc04")
    ax.bar(x,[n+p for n,p in zip(neg,[p+n for p,n in zip(pos,neu)])],0.5,
           label="Negative",color="#ea4335",bottom=[p+n for p,n in zip(pos,neu)])
    ax.set_xticks(x); ax.set_xticklabels(brands,fontsize=12)
    ax.set(ylabel="Share of text frames",ylim=(0,1.05),
           title="OCR Text Sentiment Distribution by Brand\n(VADER on on-screen text)")
    ax.legend(); fig.tight_layout()
    fig.savefig(str(output_path),dpi=130,bbox_inches="tight"); plt.show(); plt.close(fig)

def fig_boxplots(all_df, output_path):
    metrics=["brightness","saturation","colourfulness","edge_density"]
    titles=["Brightness","Saturation","Colourfulness","Edge Density"]
    brands=sorted(all_df["brand"].unique())
    fig, axes = plt.subplots(2,2,figsize=(13,9)); axes=axes.ravel()
    for ax,col,title in zip(axes,metrics,titles):
        data=[all_df[all_df["brand"]==b][col].dropna().values for b in brands]
        bp=ax.boxplot(data,labels=brands,patch_artist=True,notch=False)
        for patch,brand in zip(bp["boxes"],brands):
            patch.set_facecolor(_brand_color(brand)); patch.set_alpha(0.6)
        ax.set_title(title,fontsize=11); ax.grid(axis="y",alpha=0.3)
    fig.suptitle("Frame-Level Visual Distribution by Brand",fontsize=13,y=1.01)
    fig.tight_layout(); fig.savefig(str(output_path),dpi=130,bbox_inches="tight"); plt.show(); plt.close(fig)

def fig_wordclouds(all_df, charts_dir):
    from wordcloud import WordCloud
    for brand, grp in all_df.groupby("brand"):
        text = " ".join(grp["ocr_text"].fillna("").tolist()).lower()
        text = re.sub(r"[^a-z\s]"," ",text)
        tokens=[w for w in text.split() if len(w)>2 and w not in _STOP_WORDS]
        if not tokens: print(f"  {brand}: no OCR text for word cloud"); continue
        wc = WordCloud(width=900,height=400,background_color="white",max_words=80,
                       color_func=lambda *a,**k: _brand_color(brand)).generate(" ".join(tokens))
        fig, ax = plt.subplots(figsize=(11,5))
        ax.imshow(wc,interpolation="bilinear"); ax.axis("off")
        ax.set_title(f"{brand} — On-Screen Text Word Cloud",fontsize=13)
        fig.tight_layout()
        p = charts_dir/f"wordcloud_{brand.lower()}.png"
        fig.savefig(str(p),dpi=130,bbox_inches="tight"); plt.show(); plt.close(fig)

def fig_perception_radar(summaries, output_path):
    perc_keys = list(CLIP_PERCEPTION_VIDEO.keys())
    has_data  = any(s.get("perception_scores") for s in summaries)
    if not has_data:
        fig, ax = plt.subplots(figsize=(7,3))
        ax.text(0.5,0.5,"CLIP perception scores not available\n(requires torch + USE_DEEP_MODELS=True)",
                ha="center",va="center",fontsize=12,color="grey",transform=ax.transAxes)
        ax.axis("off"); fig.savefig(str(output_path),dpi=110,bbox_inches="tight")
        plt.show(); plt.close(fig); return
    labels = [k.replace("pxy_","").replace("_"," ").title() for k in perc_keys]
    N=len(labels); angles=[n/N*2*math.pi for n in range(N)]+[0]
    fig, ax = plt.subplots(figsize=(7,7),subplot_kw={"polar":True})
    for s in summaries:
        perc=s.get("perception_scores",{})
        vals=[perc.get(k,50) for k in perc_keys]+[perc.get(perc_keys[0],50)]
        ax.plot(angles,vals,label=s["brand"],color=_brand_color(s["brand"]),linewidth=2)
        ax.fill(angles,vals,alpha=0.1,color=_brand_color(s["brand"]))
    ax.set_xticks(angles[:-1]); ax.set_xticklabels(labels,size=9); ax.set_ylim(0,100)
    ax.set_title("CLIP Perception Scores — Brand Comparison\n(0=low, 100=high)",size=12,pad=20)
    ax.legend(loc="upper right",bbox_to_anchor=(1.35,1.1)); fig.tight_layout()
    fig.savefig(str(output_path),dpi=130,bbox_inches="tight"); plt.show(); plt.close(fig)

print("Chart functions loaded.")

## Step 4 — Video metadata preview

In [ ]:
meta_rows = [get_video_metadata(vf) for vf in video_files]
meta_df   = pd.DataFrame(meta_rows)[["brand","duration_seconds","resolution","fps","file_size_mb","total_frames"]]
print("Video metadata:\n")
display(meta_df)

## Step 5 — Run pipeline on all videos

> If `USE_DEEP_MODELS=True`, BLIP/DETR/CLIP models are downloaded (~700 MB) on first run and cached for the session.

In [ ]:
all_frames_list = []
per_video_dfs   = {}
summaries       = []

for vf in video_files:
    brand      = vf.stem.upper()
    frames_dir = FRAMES_DIR / brand.lower()
    t0         = time.time()

    df = process_video(vf, frames_dir,
                       sample_fps=SAMPLE_FPS,
                       max_frames=MAX_FRAMES,
                       verbose=True)

    # Save per-frame CSV
    csv_path = PERFRAME_DIR / f"{brand.lower()}_frames.csv"
    df.to_csv(csv_path, index=False)

    # Summarise
    s = summarise_video(df)
    summaries.append(s)

    # Save summary JSON
    jp = SUMMARIES_DIR / f"{brand.lower()}_summary.json"
    with open(jp, "w") as f: json.dump(s, f, indent=2, default=str)

    per_video_dfs[brand]  = df
    all_frames_list.append(df)
    print(f"  ✓ {brand} — {len(df)} frames in {time.time()-t0:.1f}s")

# Master CSV
master_df = pd.concat(all_frames_list, ignore_index=True)
master_df.to_csv(OUTPUT_ROOT / "master_video_analysis.csv", index=False)
print(f"\nMaster CSV: {len(master_df)} rows × {len(master_df.columns)} columns")

## Step 6 — Brand comparison table

In [ ]:
compare = pd.DataFrame([{
    "Brand":          s["brand"],
    "Duration (s)":   s.get("duration_seconds"),
    "Frames":         s.get("total_frames_analysed"),
    "Shot Cuts":      s.get("shot_cuts"),
    "Cuts/min":       s.get("cuts_per_minute"),
    "Avg Brightness": s.get("avg_brightness"),
    "Avg Saturation": s.get("avg_saturation"),
    "Avg Sharpness":  s.get("avg_sharpness"),
    "OCR Text %":     s.get("ocr_text_pct"),
    "Avg VADER":      s.get("avg_vader_compound"),
    "Dominant Theme": s.get("dominant_text_theme"),
} for s in summaries]).set_index("Brand")

display(compare)

## Step 7 — Charts

In [ ]:
print("Generating charts — they also save to Drive…\n")

# 1. Brightness timeline per brand
for brand, df in per_video_dfs.items():
    fig_brightness_timeline(df, CHARTS_DIR/f"brightness_timeline_{brand.lower()}.png")

# 2. Visual quality radar
fig_visual_radar(summaries, CHARTS_DIR/"visual_metrics_radar.png")

# 3. Shot pacing bar chart
fig_shot_pacing(summaries, CHARTS_DIR/"shot_pacing.png")

# 4. Text themes heatmap
fig_text_themes_heatmap(summaries, CHARTS_DIR/"text_themes_heatmap.png")

# 5. OCR sentiment distribution
fig_sentiment_bars(summaries, CHARTS_DIR/"sentiment_comparison.png")

# 6. Visual distribution box plots
fig_boxplots(master_df, CHARTS_DIR/"visual_distribution_boxplots.png")

# 7. Word clouds per brand
fig_wordclouds(master_df, CHARTS_DIR)

# 8. CLIP perception radar (placeholder if no torch)
fig_perception_radar(summaries, CHARTS_DIR/"perception_radar.png")

print("\nAll charts saved.")

## Step 8 — OCR keyword deep-dive

In [ ]:
for brand, df in per_video_dfs.items():
    text   = " ".join(df["ocr_text"].fillna("").tolist()).lower()
    tokens = re.sub(r"[^a-z\s]"," ",text).split()
    tokens = [w for w in tokens if len(w)>2 and w not in _STOP_WORDS]
    top20  = Counter(tokens).most_common(20)
    if not top20:
        print(f"\n{brand} — no significant OCR text found")
        continue
    print(f"\n{brand} — Top 20 OCR words:")
    for word,cnt in top20:
        bar = "█" * cnt
        print(f"  {word:<22} {bar} ({cnt})")

## Step 9 — Shot boundary detail

In [ ]:
n = len(per_video_dfs)
fig, axes = plt.subplots(n, 1, figsize=(14, 3*n), sharex=False)
if n == 1: axes = [axes]

for ax, (brand, df) in zip(axes, per_video_dfs.items()):
    c = _brand_color(brand)
    ax.plot(df["timestamp_s"], df["pixel_diff"], color=c, lw=1, alpha=0.8)
    cuts = df[df["is_shot_cut"]]
    ax.vlines(cuts["timestamp_s"], 0, df["pixel_diff"].max(),
              colors="crimson", lw=0.8, ls="--", alpha=0.5)
    ax.set_title(f"{brand} — Inter-frame pixel difference (shot cuts in red)")
    ax.set_xlabel("Time (s)"); ax.set_ylabel("Pixel diff"); ax.grid(alpha=0.2)

plt.tight_layout()
p = CHARTS_DIR / "shot_boundary_detail.png"
plt.savefig(str(p), dpi=130, bbox_inches="tight")
plt.show()
print(f"Saved: {p.name}")

## Step 10 — Strategic interpretation

> ⚠️ Exploratory only — 4 videos is not a statistically valid sample. Do not interpret as evidence of manufacturing quality or operational reliability.

In [ ]:
for s in summaries:
    brand = s["brand"]
    oem_type = "New-age OEM" if brand in ["EKA","OLECTRA","SWITCH"] else "Legacy OEM"
    cuts_pm = s.get("cuts_per_minute",0)
    pacing  = "Fast / dynamic" if cuts_pm>5 else "Moderate" if cuts_pm>2 else "Slow / deliberate"
    ocr_pct = s.get("ocr_text_pct",0)
    text_heavy = "Text-heavy" if ocr_pct>30 else "Balanced" if ocr_pct>10 else "Visuals-dominant"
    top3 = sorted(s.get("text_themes",{}).items(), key=lambda x:x[1], reverse=True)[:3]

    print(f"{'='*55}")
    print(f"  {brand}  [{oem_type}]")
    print(f"{'='*55}")
    print(f"  Duration         : {s.get('duration_seconds')}s")
    print(f"  Editing pace     : {cuts_pm} cuts/min  → {pacing}")
    print(f"  Avg brightness   : {s.get('avg_brightness')}")
    print(f"  Avg colourfulness: {s.get('avg_colourfulness')}")
    print(f"  OCR text frames  : {ocr_pct}%  → {text_heavy}")
    print(f"  VADER sentiment  : {s.get('avg_vader_compound')}")
    print(f"  Top text themes  : {', '.join(f'{t}({c})' for t,c in top3) if top3 else 'none detected'}")
    if s.get("perception_scores"):
        print(f"  CLIP perception  :")
        for k,v in s["perception_scores"].items():
            print(f"    {k.replace('pxy_',''):<22}: {v:.0f}/100")
    print()

## Step 11 — Save run summary and output index

In [ ]:
# Brand comparison CSV
compare_rows = []
for s in summaries:
    row = {k:v for k,v in s.items() if not isinstance(v,(dict,list))}
    for theme,cnt in s.get("text_themes",{}).items(): row[f"theme_{theme}"] = cnt
    compare_rows.append(row)
compare_path = SUMMARIES_DIR / "brand_comparison.csv"
pd.DataFrame(compare_rows).to_csv(compare_path, index=False)

# Run JSON
run_meta = {
    "notebook":             "05_video_analytics_colab.ipynb",
    "run_timestamp":        datetime.now().isoformat(),
    "videos_dir":           str(VIDEOS_DIR),
    "outputs_dir":          str(OUTPUT_ROOT),
    "sample_fps":           SAMPLE_FPS,
    "max_frames":           MAX_FRAMES,
    "deep_models_used":     _TORCH_AVAILABLE,
    "brands_processed":     [s["brand"] for s in summaries],
    "total_frames_analysed": int(master_df.shape[0]),
    "summaries":            summaries,
    "limitations": [
        "Exploratory only — 4 brand videos is not a statistically valid sample.",
        "Video content quality ≠ bus manufacturing quality.",
        "OCR sentiment reflects on-screen text framing, not passenger/operator experience.",
        "Shot-cut detection uses a heuristic pixel-difference threshold.",
        "CLIP/BLIP/DETR perception scores are zero-shot proxies, not ground truth.",
    ]
}
summary_path = OUTPUT_ROOT / "pipeline_run_summary.json"
with open(summary_path,"w") as f: json.dump(run_meta,f,indent=2,default=str)

# Print index
print("\nOutputs written to Google Drive:\n")
for root, dirs, files in os.walk(OUTPUT_ROOT):
    if "frames" in root and any(f.endswith(".jpg") for f in files):
        print(f"  {Path(root).relative_to(OUTPUT_ROOT)}/  [{len(files)} JPEG frames]"); continue
    for fname in sorted(files):
        fp = Path(root)/fname
        rel = fp.relative_to(OUTPUT_ROOT)
        kb  = fp.stat().st_size//1024
        print(f"  {str(rel):<55} {kb:>6} KB")

print("\n✓ All done.")